In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import os
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from torch.utils.tensorboard import SummaryWriter

# Import your custom modules
from process import get_data_loaders
from model import NeuralBKT, BKTConfig


class TrainConfig:
    def __init__(self):
        self.data_path = 'icecream_3rd.csv'
        self.block_size = 512
        self.batch_size = 32
        self.num_epochs = 50
        self.lr = 3e-4
        self.weight_decay = 0.01
        self.clip_grad_norm = 1.0
        self.patience = 5
        self.save_dir = './checkpoints'
        self.log_dir = './runs'


def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def train_one_epoch(model, loader, optimizer, scaler, cfg, device):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc="Training")
    
    for obs, output in pbar:
        obs, output = obs.to(device), output.to(device)
        
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
            _, loss = model(obs, output=output)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
        
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    
    pbar = tqdm(loader, desc="Evaluating")
    for obs, output in pbar:
        obs, output = obs.to(device), output.to(device)
        
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
            preds, loss = model(obs, output=output)
        
        total_loss += loss.item()
        mask = output[..., 0] != -1
        targets = obs[:, 1:, 0]
        preds = preds[:, :-1]
        mask = mask[:, 1:]
        
        all_preds.append(preds[mask].cpu())
        all_targets.append(targets[mask].cpu())
        
    avg_loss = total_loss / len(loader)
    val_auc = roc_auc_score(torch.cat(all_targets), torch.cat(all_preds))
    
    return avg_loss, val_auc


def main():
    cfg = TrainConfig()
    device = get_device()
    os.makedirs(cfg.save_dir, exist_ok=True)
    
    print(f"Using device: {device}")
    
    # TensorBoard
    writer = SummaryWriter(log_dir=cfg.log_dir)
    
    # Data
    print("Loading data...")
    df = pd.read_csv(cfg.data_path)
    n_skills = df['skill_id'].nunique()
    print(f"Number of skills: {n_skills}")
    
    train_loader, val_loader = get_data_loaders(
        df, block_size=cfg.block_size, batch_size=cfg.batch_size
    )
    
    # Model
    model_cfg = BKTConfig(n_skills=n_skills, block_size=cfg.block_size)
    model = NeuralBKT(model_cfg).to(device)
    
    # Optional: JIT compilation
    try:
        model = torch.compile(model)
        print("Model compiled with torch.compile")
    except Exception as e:
        print(f"torch.compile failed: {e}")
        print("Continuing without compilation")
    
    print(f"Model initialized with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")
    
    # Optimizer, scheduler, scaler
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3, verbose=True)
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
    
    # Training loop
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(cfg.num_epochs):
        print(f"Epoch {epoch+1}/{cfg.num_epochs}")
        
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, cfg, device)
        val_loss, val_auc = evaluate(model, val_loader, device)

        print(f"  Summary: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val AUC={val_auc:.4f}")
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('AUC/validation', val_auc, epoch)
        writer.add_scalar('learning_rate', optimizer.param_groups[0]['lr'], epoch)

        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            save_path = os.path.join(cfg.save_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, save_path)
            print(f"Saved best model to {save_path}")
        else:
            patience_counter += 1
            if patience_counter >= cfg.patience:
                print("Early stopping triggered")
                break
    
    writer.close()
    print(f"Training completed. Best validation loss: {best_val_loss:.4f}")

if __name__ == '__main__':
    main()